In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install pyotp mplfinance pandas_ta fyers_apiv3 pygame stable-baselines3 gymnasium shimmy optuna higher

In [3]:
import requests
import base64
from datetime import datetime, timedelta, date
from datetime import time as dt_time
import time
import threading
import pyotp
from pytz import timezone
import pandas as pd
import numpy as np
from numba import njit, prange
from urllib.parse import urlparse, parse_qs
import matplotlib.pyplot as plt
import mplfinance as mpf
import pandas_ta as ta
import pygame
import pytz
import os
import json
import sys
import gc

import gymnasium as gym
from gymnasium import spaces
from IPython.display import display, clear_output
from tqdm.notebook import tqdm

from fyers_apiv3 import fyersModel
from fyers_apiv3.FyersWebsocket import data_ws

from scipy.signal import argrelextrema
import tensorflow as tf
from collections import deque
from sklearn.preprocessing import StandardScaler

pygame 2.6.1 (SDL 2.28.4, Python 3.11.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [4]:
app_id = "TS79V3NXK1-100"
secret_key = "KQCPB0FJ74"
redirect_uri = "https://google.com"
fyers_user = "XM22383"
fyers_pin = "4628"
fyers_totp = "EAQD6K4IUYOEGPJNVE6BMPTUSDCWIOHW"
response_type = "code"
state = "sample_state"
grant_type = "authorization_code"

fyers = None
fyers_socket = None

ist_timezone = pytz.timezone("Asia/Kolkata")

# Initialize the mixer
#pygame.mixer.init()

# Load the sound file
#pygame.mixer.music.load('sounds/alert.mp3')

#Variables
ce_ltp = 0
pe_ltp = 0
index_ltp = 0
buy_sell_checked = False
ce_strike = None
pe_strike = None
ce_symbol = None
pe_symbol = None

fixed_ltp = 0
fixed_index_ltp = 0
prev_ltp = 0
target_inside = 0
target_index_inside = 0
trailing_sl_inside = 0
trailing_index_inside = 0

active_order = False

sl_hit_condition = False
total_loss = 0
total_profit = 0
overall_win = 0
overall_loss = 0
total_points = 0

unsubscribe_done = False

active_order_sleep = 1

In [5]:
session = fyersModel.SessionModel(
    client_id=app_id,
    secret_key=secret_key,
    redirect_uri=redirect_uri,
    response_type=response_type,
    grant_type=grant_type
)

def getEncodedString(string):
    string = str(string)
    base64_bytes = base64.b64encode(string.encode("ascii"))
    return base64_bytes.decode("ascii")

if session is not None:
    session.generate_authcode()

    url_send_login_otp = "https://api-t2.fyers.in/vagator/v2/send_login_otp_v2"
    res = requests.post(url=url_send_login_otp, json={"fy_id": getEncodedString(fyers_user), "app_id": "2"}).json()

    if datetime.now().second % 30 > 27:
        time.sleep(5)

    url_verify_otp = "https://api-t2.fyers.in/vagator/v2/verify_otp"
    res2 = requests.post(url=url_verify_otp, json={"request_key": res["request_key"], "otp": pyotp.TOTP(fyers_totp).now()}).json()

    ses = requests.Session()
    url_verify_otp2 = "https://api-t2.fyers.in/vagator/v2/verify_pin_v2"
    payload2 = {"request_key": res2["request_key"], "identity_type": "pin", "identifier": getEncodedString(fyers_pin)}
    res3 = ses.post(url=url_verify_otp2, json=payload2).json()

    ses.headers.update({'authorization': f"Bearer {res3['data']['access_token']}"})

    tokenurl = "https://api-t1.fyers.in/api/v3/token"
    payload3 = {
        "fyers_id": fyers_user,
        "app_id": app_id[:-4],
        "redirect_uri": redirect_uri,
        "appType": "100",
        "code_challenge": "",
        "state": "None",
        "scope": "",
        "nonce": "",
        "response_type": "code",
        "create_cookie": True
    }

    res3 = ses.post(url=tokenurl, json=payload3).json()

    url = res3['Url']
    parsed = urlparse(url)
    auth_code = parse_qs(parsed.query)['auth_code'][0]

    session.set_token(auth_code)

    auth_response = session.generate_token()
    access_token = auth_response["access_token"]

    fyers = fyersModel.FyersModel(client_id=app_id, token=access_token)

    ws_token = app_id + ":" + access_token
    fyers_socket = data_ws.FyersDataSocket(access_token=ws_token, log_path="")

pd.DataFrame(fyers.get_profile())

,s,code,message,data
fy_id,ok,200,,XM22383
name,ok,200,,MARSHAL TUDU
image,ok,200,,https://myaccount-docs-prod.fyers.in/Profile_P...
display_name,ok,200,,None
pin_change_date,ok,200,,25-09-2023 17:16:16
email_id,ok,200,,iammarshal22@gmail.com
pwd_change_date,ok,200,,01-06-2022 20:36:31
PAN,ok,200,,---------
mobile_number,ok,200,,8458060663
totp,ok,200,,True


In [6]:
def fetch_candle_data(number, index_symbol, interval_minutes):
    while True:
        try:
            today = date.today()
            yesterday = today - timedelta(number)

            data = {
                "symbol": index_symbol,
                "resolution": interval_minutes,
                "date_format": "1",
                "range_from": yesterday,
                "range_to": today,
                "cont_flag": "1"
            }

            result = fyers.history(data=data)

            if result is not None:
                train_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                return train_df
        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [7]:
def fetch_train_candle_data(days_count, index_symbol, interval_minutes):
    train_df = pd.DataFrame()

    while True:
        try:
            date_increment = 100
            for i in range(days_count):
                today = date.today() - timedelta(date_increment)
                yesterday = today - timedelta(100)

                data = {
                    "symbol": index_symbol,
                    "resolution": interval_minutes,
                    "date_format": "1",
                    "range_from": yesterday,
                    "range_to": today,
                    "cont_flag": "1"
                }

                result = fyers.history(data=data)

                if result is not None:
                    temp_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                    train_df = pd.concat([temp_df, train_df], ignore_index=True)

                date_increment += 100

            if train_df is not None:
                return train_df

        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [8]:
index_mapping = {
    # Indices
    "Nifty": {
        "symbol": "NSE:NIFTY50-INDEX",
        "quantity": 75
    },
    "Bank_Nifty": {
        "symbol": "NSE:NIFTYBANK-INDEX",
        "quantity": 30
    },
    "Finnifty": {
        "symbol": "NSE:FINNIFTY-INDEX",
        "quantity": 40
    },
    "Sensex": {
        "symbol": "BSE:SENSEX-INDEX",
        "quantity": 20
    },
    "Bankex": {
        "symbol": "BSE:BANKEX-INDEX",
        "quantity": 15
    },

    # Complete Nifty 50 Stocks

    "Adani_Enterprises_Ltd": {
        "symbol": "NSE:ADANIENT-EQ",
        "quantity": 1
    },
    "Adani_Ports_&_SEZ_Ltd": {
        "symbol": "NSE:ADANIPORTS-EQ",
        "quantity": 1
    },
    "Apollo_Hospitals_Enterprise_Ltd": {
        "symbol": "NSE:APOLLOHOSP-EQ",
        "quantity": 1
    },
    "Asian_Paints_Ltd": {
        "symbol": "NSE:ASIANPAINT-EQ",
        "quantity": 1
    },
    "Axis_Bank_Ltd": {
        "symbol": "NSE:AXISBANK-EQ",
        "quantity": 1
    },
    "Bajaj_Auto_Ltd": {
        "symbol": "NSE:BAJAJ-AUTO-EQ",
        "quantity": 1
    },
    "Bajaj_Finance_Ltd": {
        "symbol": "NSE:BAJFINANCE-EQ",
        "quantity": 1
    },
    "Bajaj_Finserv_Ltd": {
        "symbol": "NSE:BAJAJFINSV-EQ",
        "quantity": 1
    },
    "Bharat_Electronics_Ltd": {
        "symbol": "NSE:BEL-EQ",
        "quantity": 1
    },
    "Bharti_Airtel_Ltd": {
        "symbol": "NSE:BHARTIARTL-EQ",
        "quantity": 1
    },
    "BPCL_Ltd": {
        "symbol": "NSE:BPCL-EQ",
        "quantity": 1
    },
    "Cipla_Ltd": {
        "symbol": "NSE:CIPLA-EQ",
        "quantity": 1
    },
    "Coal_India_Ltd": {
        "symbol": "NSE:COALINDIA-EQ",
        "quantity": 1
    },
    "Dr_Reddy's_Laboratories_Ltd": {
        "symbol": "NSE:DRREDDY-EQ",
        "quantity": 1
    },
    "Eicher_Motors_Ltd": {
        "symbol": "NSE:EICHERMOT-EQ",
        "quantity": 1
    },
    "Grasim_Industries_Ltd": {
        "symbol": "NSE:GRASIM-EQ",
        "quantity": 1
    },
    "HCL_Technologies_Ltd": {
        "symbol": "NSE:HCLTECH-EQ",
        "quantity": 1
    },
    "HDFC_Bank_Ltd": {
        "symbol": "NSE:HDFCBANK-EQ",
        "quantity": 1
    },
    "HDFC_Life_Insurance_Company_Ltd": {
        "symbol": "NSE:HDFCLIFE-EQ",
        "quantity": 1
    },
    "Hero_MotoCorp_Ltd": {
        "symbol": "NSE:HEROMOTOCO-EQ",
        "quantity": 1
    },
    "Hindalco_Industries_Ltd": {
        "symbol": "NSE:HINDALCO-EQ",
        "quantity": 1
    },
    "Hindustan_Unilever_Ltd": {
        "symbol": "NSE:HINDUNILVR-EQ",
        "quantity": 1
    },
    "ICICI_Bank_Ltd": {
        "symbol": "NSE:ICICIBANK-EQ",
        "quantity": 1
    },
    "IndusInd_Bank_Ltd": {
        "symbol": "NSE:INDUSINDBK-EQ",
        "quantity": 1
    },
    "Infosys_Ltd": {
        "symbol": "NSE:INFY-EQ",
        "quantity": 1
    },
    "Indian_Oil_Corporation_Ltd": {
        "symbol": "NSE:IOC-EQ",
        "quantity": 1
    },
    "JSW_Steel_Ltd": {
        "symbol": "NSE:JSWSTEEL-EQ",
        "quantity": 1
    },
    "Kotak_Mahindra_Bank_Ltd": {
        "symbol": "NSE:KOTAKBANK-EQ",
        "quantity": 1
    },
    "Larsen_&_Toubro_Ltd": {
        "symbol": "NSE:LT-EQ",
        "quantity": 1
    },
    "Mahindra_&_Mahindra_Ltd": {
        "symbol": "NSE:M&M-EQ",
        "quantity": 1
    },
    "Maruti_Suzuki_India_Ltd": {
        "symbol": "NSE:MARUTI-EQ",
        "quantity": 1
    },
    "Nestle_India_Ltd": {
        "symbol": "NSE:NESTLEIND-EQ",
        "quantity": 1
    },
    "NTPC_Ltd": {
        "symbol": "NSE:NTPC-EQ",
        "quantity": 1
    },
    "ONGC_Ltd": {
        "symbol": "NSE:ONGC-EQ",
        "quantity": 1
    },
    "Power_Grid_Corporation_of_India_Ltd": {
        "symbol": "NSE:POWERGRID-EQ",
        "quantity": 1
    },
    "Reliance_Industries_Ltd": {
        "symbol": "NSE:RELIANCE-EQ",
        "quantity": 1
    },
    "State_Bank_of_India_(SBI)_Ltd": {
        "symbol": "NSE:SBIN-EQ",
        "quantity": 1
    },
    "SBI_Life_Insurance_Company_Ltd": {
        "symbol": "NSE:SBILIFE-EQ",
        "quantity": 1
    },
    "Sun_Pharmaceutical_Industries_Ltd": {
        "symbol": "NSE:SUNPHARMA-EQ",
        "quantity": 1
    },
    "Tata_Consumer_Products_Ltd": {
        "symbol": "NSE:TATACONSUM-EQ",
        "quantity": 1
    },
    "Tata_Motors_Ltd": {
        "symbol": "NSE:TATAMOTORS-EQ",
        "quantity": 1
    },
    "Tata_Steel_Ltd": {
        "symbol": "NSE:TATASTEEL-EQ",
        "quantity": 1
    },
    "TCS_Ltd": {
        "symbol": "NSE:TCS-EQ",
        "quantity": 1
    },
    "Tech_Mahindra_Ltd": {
        "symbol": "NSE:TECHM-EQ",
        "quantity": 1
    },
    "Titan_Company_Ltd": {
        "symbol": "NSE:TITAN-EQ",
        "quantity": 1
    },
    "Trent_Ltd": {
        "symbol": "NSE:TRENT-EQ",
        "quantity": 1
    },
    "UltraTech_Cement_Ltd": {
        "symbol": "NSE:ULTRACEMCO-EQ",
        "quantity": 1
    },
    "Wipro_Ltd": {
        "symbol": "NSE:WIPRO-EQ",
        "quantity": 1
    }
}

In [9]:
# Historical config for fetching and saving raw data
history_config = {
    "timeframes": [1, 2, 3, 5, 10, 15, 20, 30, 45, 60, 120, 180, 240],
    "colab_folder_path": "/content/drive/MyDrive/historical_data",
    "colab_folder_path_processed": "processed_data",
    "local_folder_path": "historical_data",
    "local_folder_path_processed": "C:\\Users\\iamma\\OneDrive\\Desktop\\Marshal\\processed_data",
    "bar_limit": 15  # parameter for fetch_train_candle_data
}

In [10]:
if "google.colab" in sys.modules:
    folder_path = history_config["colab_folder_path"]
else:
    folder_path = history_config["local_folder_path"]

In [11]:
# Ensure the folder exists
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

In [12]:
# Main loop to fetch and save data
for instrument_name, info in index_mapping.items():
    for tf in history_config["timeframes"]:
        file_name = f"{instrument_name}_{tf}.csv"
        file_path = os.path.join(folder_path, file_name)

        # Skip if file already exists
        if os.path.exists(file_path):
            print(f"File already exists for {instrument_name} timeframe {tf}. Skipping.")
            continue

        # Fetch data for given bar_limit, symbol, and timeframe
        df = fetch_train_candle_data(history_config["bar_limit"], info["symbol"], tf)

        # Save if data is not empty
        if not df.empty:
            df.to_csv(file_path, index=False)
            print(f"Saved data for {instrument_name} timeframe {tf} at {file_path}")
        else:
            print(f"No data returned for {instrument_name} timeframe {tf}.")

File already exists for Nifty timeframe 1. Skipping.
File already exists for Nifty timeframe 2. Skipping.
File already exists for Nifty timeframe 3. Skipping.
File already exists for Nifty timeframe 5. Skipping.
File already exists for Nifty timeframe 10. Skipping.
File already exists for Nifty timeframe 15. Skipping.
File already exists for Nifty timeframe 20. Skipping.
File already exists for Nifty timeframe 30. Skipping.
File already exists for Nifty timeframe 45. Skipping.
File already exists for Nifty timeframe 60. Skipping.
File already exists for Nifty timeframe 120. Skipping.
File already exists for Nifty timeframe 180. Skipping.
File already exists for Nifty timeframe 240. Skipping.
File already exists for Bank_Nifty timeframe 1. Skipping.
File already exists for Bank_Nifty timeframe 2. Skipping.
File already exists for Bank_Nifty timeframe 3. Skipping.
File already exists for Bank_Nifty timeframe 5. Skipping.
File already exists for Bank_Nifty timeframe 10. Skipping.
File alr

In [13]:
@njit
def label_signals_jit(close_arr, high_arr, low_arr, target_arr, stoploss_arr):
    # close_arr, high_arr, low_arr, target_arr, stoploss_arr are float arrays
    # Returns:
    # signals, entry_prices, exit_prices, candles_to_profit, candles_to_loss
    # each as float64 or int64 arrays

    n = len(close_arr)
    signals = np.zeros(n, dtype=np.int64)
    entry_prices = np.zeros(n, dtype=np.float64)
    exit_prices = np.zeros(n, dtype=np.float64)
    candles_to_profit = np.zeros(n, dtype=np.int64)
    candles_to_loss = np.zeros(n, dtype=np.int64)

    for i in range(n - 1):
        entry_price = close_arr[i]
        target = target_arr[i]
        stop_loss = stoploss_arr[i]

        buy_target_price = entry_price + target
        buy_sl_price = entry_price - stop_loss
        sell_target_price = entry_price - target
        sell_sl_price = entry_price + stop_loss

        signal_found = False
        for offset in range(i+1, n):
            future_high = high_arr[offset]
            future_low = low_arr[offset]

            triggers = []
            # Check if buy target/SL triggered
            if future_high >= buy_target_price:
                triggers.append((1, offset - i))  # (Signal=1 => buy_target)
            if future_low <= buy_sl_price:
                triggers.append((2, offset - i))  # (Signal=2 => buy_sl)
            # Check if sell target/SL triggered
            if future_low <= sell_target_price:
                triggers.append((3, offset - i))  # (Signal=3 => sell_target)
            if future_high >= sell_sl_price:
                triggers.append((4, offset - i))  # (Signal=4 => sell_sl)

            # If there are triggers, pick the earliest by priority
            if triggers:
                # Priority: 1 => buy_target, 3 => sell_target, 2 => buy_sl, 4 => sell_sl
                triggers.sort(key=lambda x: x[0])
                first_trigger, candle_count = triggers[0]
                signals[i] = first_trigger
                entry_prices[i] = entry_price

                if first_trigger == 1:  # buy_target
                    exit_prices[i] = buy_target_price
                    candles_to_profit[i] = candle_count
                elif first_trigger == 2:  # buy_sl
                    exit_prices[i] = buy_sl_price
                    candles_to_loss[i] = candle_count
                elif first_trigger == 3:  # sell_target
                    exit_prices[i] = sell_target_price
                    candles_to_profit[i] = candle_count
                elif first_trigger == 4:  # sell_sl
                    exit_prices[i] = sell_sl_price
                    candles_to_loss[i] = candle_count

                signal_found = True
                break

        if not signal_found:
            signals[i] = 0
            entry_prices[i] = entry_price

    return signals, entry_prices, exit_prices, candles_to_profit, candles_to_loss

# %% [code]
class FullFeaturePipeline:
    def __init__(self, df: pd.DataFrame):
        # df must have columns [datetime, open, high, low, close, (volume optional)]
        self.df = df.copy()

    def preprocess_datetime(self):
        # Convert Unix timestamp to local time (Asia/Kolkata)
        ist = timezone('Asia/Kolkata')

        self.df['datetime'] = pd.to_datetime(self.df['datetime'], unit='s')
        self.df['datetime'] = (self.df['datetime']
                               .dt.tz_localize('UTC')
                               .dt.tz_convert(ist)
                               .dt.tz_localize(None))

        # Check for duplicates or missing
        if self.df['datetime'].duplicated().any() or self.df['datetime'].isnull().any():
            raise ValueError("The 'datetime' column contains duplicates or missing values.")

        # Sort and set as index
        self.df.sort_values('datetime', inplace=True)
        self.df.set_index('datetime', inplace=True)
        return self

    def clean_data(self):
        # Drop volume if zero or NaN
        if 'volume' in self.df.columns:
            if self.df['volume'].isnull().any() or (self.df['volume'] == 0).any():
                self.df.drop('volume', axis=1, inplace=True, errors='ignore')
        return self

    def add_indicator_features(self):
        # Using pandas_ta ATR
        self.df['atr_14'] = ta.atr(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            length=14
        )
        self.df['atr_14'] = round(self.df['atr_14'], 2)

        return self

    def add_time_features(self):
        # Some cyclical time features
        self.df.sort_index(inplace=True)
        self.df['hour'] = self.df.index.hour.astype(float)
        self.df['day_of_week'] = self.df.index.dayofweek.astype(float)

        self.df['hour_sin'] = np.sin(2 * np.pi * self.df['hour'] / 24.0)
        self.df['hour_cos'] = np.cos(2 * np.pi * self.df['hour'] / 24.0)
        self.df['dow_sin'] = np.sin(2 * np.pi * self.df['day_of_week'] / 7.0)
        self.df['dow_cos'] = np.cos(2 * np.pi * self.df['day_of_week'] / 7.0)
        return self

    def add_adaptive_targets_and_stops(self):
        # Adjust the multiples as per your strategy
        self.df['Target'] = 2 * self.df['atr_14']
        self.df['StopLoss'] = 1 * self.df['atr_14']
        return self

    def label_signals(self):
        close_arr = self.df['close'].values
        high_arr = self.df['high'].values
        low_arr = self.df['low'].values
        target_arr = self.df['Target'].values
        stoploss_arr = self.df['StopLoss'].values

        signals, entry_prices, exit_prices, ctp, ctl = label_signals_jit(
            close_arr.astype(np.float64),
            high_arr.astype(np.float64),
            low_arr.astype(np.float64),
            target_arr.astype(np.float64),
            stoploss_arr.astype(np.float64),
        )

        self.df['Signal'] = signals
        self.df['Entry Price'] = entry_prices
        self.df['Exit Price'] = exit_prices
        self.df['candles_to_profit'] = ctp
        self.df['candles_to_loss'] = ctl

        return self

    def run_pipeline(self):
        (
            self.preprocess_datetime()
                .clean_data()
                .add_indicator_features()
                .add_time_features()
                .add_adaptive_targets_and_stops()
                .label_signals()
        )
        return self

    def get_processed_df(self):
        # Drop rows that have NaN from newly added features
        self.df.dropna(axis=0, how='any', inplace=True)

        # Drop 'Entry Price' and 'Exit Price' if you do not want them in final features
        '''if 'Entry Price' in self.df.columns:
            self.df.drop('Entry Price', axis=1, inplace=True)
        if 'Exit Price' in self.df.columns:
            self.df.drop('Exit Price', axis=1, inplace=True)'''

        return self.df

In [14]:
if "google.colab" in sys.modules:
    processed_folder_path = history_config["colab_folder_path_processed"]
else:
    processed_folder_path = history_config["local_folder_path_processed"]

if not os.path.exists(processed_folder_path):
    os.makedirs(processed_folder_path)

In [15]:
# We'll store (instrument_name, timeframe, processed_df) tuples in this list
temp_data_list = []

for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        file_path = os.path.join(folder_path, file)

        # Split the filename on the last underscore to accommodate multiple underscores in the instrument name
        instrument_name, timeframe_str = file.replace(".csv", "").rsplit("_", 1)
        timeframe = int(timeframe_str)

        processed_file_name = f"{instrument_name}_{timeframe}_processed.csv"
        processed_file_path = os.path.join(processed_folder_path, processed_file_name)

        # If processed file exists, skip reprocessing
        if os.path.exists(processed_file_path):
            print(f"Processed file already exists for {instrument_name} {timeframe}M. Skipping.")
        else:
            # Process it once, then drop from memory
            raw_df = pd.read_csv(file_path).drop_duplicates(subset='datetime', keep='first')
            pipeline = FullFeaturePipeline(raw_df)
            pipeline.run_pipeline()
            processed_df = pipeline.get_processed_df()

            processed_df.to_csv(processed_file_path, index=True, index_label='datetime')
            print(f"Processed and saved data for {instrument_name} {timeframe}M at {processed_file_path}")

            # Cleanup to free memory
            del raw_df
            del processed_df
            gc.collect()

        # We always add the path so we know it exists (either newly processed or previously)
        temp_data_list.append((instrument_name, timeframe, processed_file_path))

Processed file already exists for Adani_Enterprises_Ltd 10M. Skipping.
Processed file already exists for Adani_Ports_&_SEZ_Ltd 240M. Skipping.
Processed file already exists for Adani_Enterprises_Ltd 3M. Skipping.
Processed file already exists for Adani_Enterprises_Ltd 120M. Skipping.
Processed file already exists for Adani_Enterprises_Ltd 2M. Skipping.
Processed file already exists for Adani_Ports_&_SEZ_Ltd 180M. Skipping.
Processed file already exists for Adani_Enterprises_Ltd 5M. Skipping.
Processed file already exists for Adani_Enterprises_Ltd 45M. Skipping.
Processed file already exists for Adani_Enterprises_Ltd 30M. Skipping.
Processed file already exists for Adani_Enterprises_Ltd 60M. Skipping.
Processed file already exists for Adani_Ports_&_SEZ_Ltd 15M. Skipping.
Processed file already exists for Adani_Ports_&_SEZ_Ltd 1M. Skipping.
Processed file already exists for Adani_Enterprises_Ltd 15M. Skipping.
Processed file already exists for Adani_Ports_&_SEZ_Ltd 120M. Skipping.
Proces

In [16]:
# Sort by timeframe descending, then by instrument name alphabetically
temp_data_list.sort(key=lambda x: (-x[1], x[0]))

instruments_config = []
for instrument_name, timeframe, csv_path in temp_data_list:
    print(instrument_name)
    lot_size = index_mapping[instrument_name]["quantity"]
    dynamic_name = f"{instrument_name.upper()}_{timeframe}M"

    # Instead of storing the entire DataFrame, just keep the path
    instruments_config.append({
        "name": dynamic_name,
        "file_path": csv_path,
        "lot_size": lot_size,
        "transaction_cost": 20.0  # default brokerage
    })

Adani_Enterprises_Ltd
Adani_Ports_&_SEZ_Ltd
Apollo_Hospitals_Enterprise_Ltd
Asian_Paints_Ltd
Axis_Bank_Ltd
BPCL_Ltd
Bajaj_Auto_Ltd
Bajaj_Finance_Ltd
Bajaj_Finserv_Ltd
Bank_Nifty
Bankex
Bharat_Electronics_Ltd
Bharti_Airtel_Ltd
Cipla_Ltd
Coal_India_Ltd
Dr_Reddy's_Laboratories_Ltd
Eicher_Motors_Ltd
Finnifty
Grasim_Industries_Ltd
HCL_Technologies_Ltd
HDFC_Bank_Ltd
HDFC_Life_Insurance_Company_Ltd
Hero_MotoCorp_Ltd
Hindalco_Industries_Ltd
Hindustan_Unilever_Ltd
ICICI_Bank_Ltd
Indian_Oil_Corporation_Ltd
IndusInd_Bank_Ltd
Infosys_Ltd
JSW_Steel_Ltd
Kotak_Mahindra_Bank_Ltd
Larsen_&_Toubro_Ltd
Mahindra_&_Mahindra_Ltd
Maruti_Suzuki_India_Ltd
NTPC_Ltd
Nestle_India_Ltd
Nifty
ONGC_Ltd
Power_Grid_Corporation_of_India_Ltd
Reliance_Industries_Ltd
SBI_Life_Insurance_Company_Ltd
Sensex
State_Bank_of_India_(SBI)_Ltd
Sun_Pharmaceutical_Industries_Ltd
TCS_Ltd
Tata_Consumer_Products_Ltd
Tata_Motors_Ltd
Tata_Steel_Ltd
Tech_Mahindra_Ltd
Titan_Company_Ltd
Trent_Ltd
UltraTech_Cement_Ltd
Wipro_Ltd
Adani_Enterprise

In [17]:
# Now build the main config dictionary
config = {
    "instruments": instruments_config,
    "window_size": 30,
    "unrealized_pnl_weight": 0.1,
    "time_penalty_weight": 0.001,
    "volatility_penalty_weight": 0.05,
    "target_bonus": 0.5,
    "sl_penalty": 0.5,
    "misalignment_penalty": 0.5,
    "missed_opportunity_penalty": 0.05,

    # Trailing Stop Parameters
    "enable_trailing_sl": True,
    "trailing_sl_mode": "factor",   # can be "fixed", "percentage", or "factor"
    "trailing_sl_amount": 20.0,
    "trailing_sl_percent": 0.01,
    "trailing_factor": 0.5,

    "initial_cap_multiplier": 10,   # factor for initial capital
    "normalize_data": True          # whether to normalize data for the agent's observation
}

# Margin constants
MARGIN_FRACTION = 0.05
MARGIN_CALL_THRESHOLD = 0.5

In [18]:
# Print a summary of the dynamically created instruments
print("Final Config Instruments:")
for inst in config["instruments"]:
    print(f"{inst['name']}: lot_size={inst['lot_size']}, df_length={len(inst['file_path'])}, brokerage={inst['transaction_cost']}")

Final Config Instruments:
ADANI_ENTERPRISES_LTD_240M: lot_size=1, df_length=54, brokerage=20.0
ADANI_PORTS_&_SEZ_LTD_240M: lot_size=1, df_length=54, brokerage=20.0
APOLLO_HOSPITALS_ENTERPRISE_LTD_240M: lot_size=1, df_length=64, brokerage=20.0
ASIAN_PAINTS_LTD_240M: lot_size=1, df_length=49, brokerage=20.0
AXIS_BANK_LTD_240M: lot_size=1, df_length=46, brokerage=20.0
BPCL_LTD_240M: lot_size=1, df_length=41, brokerage=20.0
BAJAJ_AUTO_LTD_240M: lot_size=1, df_length=47, brokerage=20.0
BAJAJ_FINANCE_LTD_240M: lot_size=1, df_length=50, brokerage=20.0
BAJAJ_FINSERV_LTD_240M: lot_size=1, df_length=50, brokerage=20.0
BANK_NIFTY_240M: lot_size=30, df_length=43, brokerage=20.0
BANKEX_240M: lot_size=15, df_length=39, brokerage=20.0
BHARAT_ELECTRONICS_LTD_240M: lot_size=1, df_length=55, brokerage=20.0
BHARTI_AIRTEL_LTD_240M: lot_size=1, df_length=50, brokerage=20.0
CIPLA_LTD_240M: lot_size=1, df_length=42, brokerage=20.0
COAL_INDIA_LTD_240M: lot_size=1, df_length=47, brokerage=20.0
DR_REDDY'S_LABOR

In [19]:
def get_dataframe_by_name(instrument_name, config):
    # instrument_name is something like "BANKNIFTY_5M", "NIFTY50_15M", etc.
    for instrument in config["instruments"]:
        if instrument["name"] == instrument_name:
            return pd.read_csv(instrument["file_path"], parse_dates=['datetime'], index_col='datetime')
    return None


my_df = get_dataframe_by_name("BANKNIFTY_5M", config)

my_df

In [20]:
class Position:
    # Holds all necessary info about the current position (long or short).
    def __init__(self):
        self.reset()

    def reset(self):
        self.status = "flat"             # "flat", "long", or "short"
        self.entry_price = None
        self.sl_points = None            # For initial SL
        self.target_points = None
        self.direction = 0               # +1 for long, -1 for short
        self.quantity = 0
        self.time_in_trade = 0
        self.margin_used = 0.0           # Tracks margin used for the current position

        # Trailing SL related
        self.trailing_sl_price = None    # Absolute trailing SL price
        self.highest_price = None        # For tracking highest price since entry (long)
        self.lowest_price = None         # For tracking lowest price since entry (short)

    def update_unrealized_pnl(self, current_price: float) -> float:
        if self.status == "flat":
            return 0.0
        return (current_price - self.entry_price) * self.direction * self.quantity

    def open(self, entry_price: float, sl_points: float, target_points: float,
             direction: int, quantity: int):
        self.status = "long" if direction == 1 else "short"
        self.entry_price = entry_price
        self.sl_points = sl_points
        self.target_points = target_points
        self.direction = direction
        self.quantity = quantity
        self.time_in_trade = 0

        # Initialize trailing stop to the initial SL directly
        if direction == 1:
            self.trailing_sl_price = entry_price - sl_points
            self.highest_price = entry_price
            self.lowest_price = None
        else:
            self.trailing_sl_price = entry_price + sl_points
            self.highest_price = None
            self.lowest_price = entry_price

In [21]:
class TradingEnv(gym.Env):
    def __init__(self, env_config):
        super(TradingEnv, self).__init__()
        self.config = env_config
        self.total_reward = 0.0

        csv_path = self.config["file_path"]
        self.df =pd.read_csv(csv_path, parse_dates=['datetime'], index_col='datetime')

        # Window size
        self.window_size = self.config["window_size"]

        # Guidance columns (Signal, candles_to_profit, candles_to_loss)
        self.df_guidance = self.df[['Signal','candles_to_profit','candles_to_loss']].copy()

        # Raw data (needed for environment logic, reward calc, etc.)
        # Dropping the columns that are specifically for guidance
        self.df_raw = self.df.drop(columns=['Signal','candles_to_profit','candles_to_loss']).copy()

        # Decide which columns to exclude from agent's observation
        self.exclude_from_obs = ["StopLoss", "Target"]

        # We'll drop the exclude_from_obs columns from df_raw here
        self.df_obs = self.df_raw.drop(columns=self.exclude_from_obs, errors="ignore").copy()

        self.normalize_data = self.config.get("normalize_data", False)
        if self.normalize_data:
            self.scaler = StandardScaler()
            # Fit only on the columns the agent *will* observe
            self.scaler.fit(self.df_obs.values)

            # Transform the observation DataFrame
            self.df_norm = pd.DataFrame(
                self.scaler.transform(self.df_obs.values),
                columns=self.df_obs.columns,
                index=self.df_obs.index
            )
        else:
            self.df_norm = self.df_obs.copy()

        # Other environment parameters
        self.lot_size = self.config["lot_size"]
        self.transaction_cost = self.config["transaction_cost"]
        self.instrument_name = self.config.get("name", "Unknown")

        # Instead of max(high)*3, use margin-based approach for initial capital
        multiplier = self.config.get("initial_cap_multiplier", 3)
        self.initial_capital = self.df_raw['high'].max() * self.lot_size * MARGIN_FRACTION * multiplier
        self.current_capital = None
        self.current_step = None
        self.position = Position()
        self.trade_log = []

        # Action space: 0 = Hold, 1 = Buy, 2 = Sell, 3 = Close
        self.action_space = spaces.Discrete(4)

        # Prepare observation space based on the shape of df_norm
        n_features = len(self.df_norm.columns)
        self.observation_space = spaces.Dict({
            "market": spaces.Box(-np.inf, np.inf, shape=(n_features,), dtype=np.float32),
            "position": spaces.Box(-np.inf, np.inf, shape=(4,), dtype=np.float32),
            "history": spaces.Box(-np.inf, np.inf, shape=(self.window_size, n_features), dtype=np.float32),
            "instrument_info": spaces.Box(-np.inf, np.inf, shape=(2,), dtype=np.float32)
        })

        # Reward shaping parameters
        self.unrealized_pnl_weight = self.config.get("unrealized_pnl_weight", 0.1)
        self.time_penalty_weight = self.config.get("time_penalty_weight", 0.02)
        self.volatility_penalty_weight = self.config.get("volatility_penalty_weight", 0.1)

        # Guidance reward parameters
        self.target_bonus = self.config.get("target_bonus", 0.5)
        self.sl_penalty = self.config.get("sl_penalty", 0.5)
        self.misalignment_penalty = self.config.get("misalignment_penalty", 0.5)
        self.missed_opportunity_penalty = self.config.get("missed_opportunity_penalty", 0.05)

        # Trailing SL config
        self.enable_trailing_sl = self.config.get("enable_trailing_sl", True)
        self.trailing_sl_mode = self.config.get("trailing_sl_mode", "fixed")
        self.trailing_sl_amount = self.config.get("trailing_sl_amount", 20.0)
        self.trailing_sl_percent = self.config.get("trailing_sl_percent", 0.01)
        self.trailing_factor = self.config.get("trailing_factor", 0.5)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = self.window_size
        self.position.reset()
        self.current_capital = self.initial_capital
        self.trade_log = []
        self.total_reward = 0.0
        return self._get_observation(), {}

    def step(self, action: int):
        current_data = self.df_raw.iloc[self.current_step]
        guidance_row = self.df_guidance.iloc[self.current_step]
        reward = 0.0

        # -----------------------------
        # Immediate Guidance Reward (Only When Flat)
        # -----------------------------
        if self.position.status == "flat":
            signal = guidance_row['Signal']
            candles_to_profit = guidance_row['candles_to_profit'] if guidance_row['candles_to_profit'] > 0 else 1.0
            candles_to_loss = guidance_row['candles_to_loss'] if guidance_row['candles_to_loss'] > 0 else 1.0

            if signal == 1:  # Target for long
                correct_action = 1
                guidance_reward = self.target_bonus / candles_to_profit
            elif signal == 2:  # SL for long
                correct_action = 3
                guidance_reward = -self.sl_penalty / candles_to_loss
            elif signal == 3:  # Target for short
                correct_action = 2
                guidance_reward = self.target_bonus / candles_to_profit
            elif signal == 4:  # SL for short
                correct_action = 3
                guidance_reward = -self.sl_penalty / candles_to_loss
            else:
                correct_action = None
                guidance_reward = 0.0

            if action == correct_action:
                reward += guidance_reward
            else:
                reward -= self.misalignment_penalty

        # -----------------------------
        # If we have a position, check margin call conditions
        # -----------------------------
        if self.position.status != "flat":
            if self._check_margin_call(current_data['close']):
                reward += self._close_position(current_data, "Margin Call")

        # -----------------------------
        # Update Trailing SL (if enabled and position is open)
        # -----------------------------
        if self.enable_trailing_sl and self.position.status != "flat":
            self._update_trailing_sl(current_data)

        # -----------------------------
        # Check for SL or Trailing SL or Target Hit (intra-candle logic)
        # -----------------------------
        hit_sl, fill_price = self._check_sl_hit(current_data)
        if hit_sl:
            reward += self._close_position(current_data, "SL/Trailing SL Hit", override_exit_price=fill_price)

        # -----------------------------
        # Execute Action
        # -----------------------------
        self._execute_action(action, current_data)

        # -----------------------------
        # Time in trade increment & shaping reward
        # -----------------------------
        if self.position.status != "flat":
            self.position.time_in_trade += 1
            unrealized_pnl = self.position.update_unrealized_pnl(current_data['close'])
            reward += self._calculate_reward(unrealized_pnl, current_data)

        # -----------------------------
        # Advance environment
        # -----------------------------
        self.current_step += 1
        if self.current_step >= len(self.df_raw) or self.current_capital <= 0:
            done = True
            # Force close if still in position
            if self.position.status != "flat":
                reward += self._close_position(current_data, "Force Close")
        else:
            done = False

        self.total_reward += reward
        return self._get_observation(), reward, done, False, {}

    def _get_observation(self) -> dict:
        # Here we provide normalized data to the agent, but raw data is used internally.
        idx = min(self.current_step, len(self.df_norm) - 1)

        # Normalized features
        market_features = self.df_norm.iloc[idx].values.astype(np.float32)

        # Position context
        current_close = self.df_raw.iloc[idx]['close']
        unrealized_pnl = self.position.update_unrealized_pnl(current_close)
        normalized_pnl = unrealized_pnl / self.initial_capital
        position_context = np.array([
            1.0 if self.position.status == "long" else 0.0,
            1.0 if self.position.status == "short" else 0.0,
            normalized_pnl,
            self.position.time_in_trade / 100.0
        ], dtype=np.float32)

        # History window (normalized)
        history_start = max(0, idx - self.window_size)
        history_slice = self.df_norm.iloc[history_start:idx]
        history = history_slice.values.astype(np.float32)
        if len(history) < self.window_size:
            padding = np.zeros((self.window_size - len(history), self.df_norm.shape[1]), dtype=np.float32)
            history = np.vstack([padding, history])

        # Instrument info
        normalized_lot_size = self.lot_size / 1000.0
        normalized_transaction_cost = self.transaction_cost / 100.0
        instrument_info = np.array([normalized_lot_size, normalized_transaction_cost], dtype=np.float32)

        return {
            "market": market_features,
            "position": position_context,
            "history": history,
            "instrument_info": instrument_info
        }

    def _execute_action(self, action: int, current_data: pd.Series):
        price = current_data['open']

        # Close action
        if action == 3 and self.position.status != "flat":
            self._close_position(current_data, "Manual Close")

        # Buy action
        elif action == 1:
            if self.position.status == "short":
                self._close_position(current_data, "Reverse Close")
            margin_cost = price * self.lot_size * MARGIN_FRACTION
            if self.position.status == "flat" and self.current_capital >= (margin_cost + self.transaction_cost):
                self._open_position("long", current_data)

        # Sell action
        elif action == 2:
            if self.position.status == "long":
                self._close_position(current_data, "Reverse Close")
            margin_cost = price * self.lot_size * MARGIN_FRACTION
            if self.position.status == "flat" and self.current_capital >= (margin_cost + self.transaction_cost):
                self._open_position("short", current_data)

        # If action == 0 => Hold/do nothing

    def _open_position(self, direction: str, current_data: pd.Series):
        entry_price = current_data['open']
        margin_cost = entry_price * self.lot_size * MARGIN_FRACTION
        self.current_capital -= (margin_cost + self.transaction_cost)
        self.position.margin_used = margin_cost

        self.position.open(
            entry_price=entry_price,
            sl_points=current_data['StopLoss'],
            target_points=current_data['Target'],
            direction=1 if direction == "long" else -1,
            quantity=self.lot_size
        )

        self.trade_log.append({
            "entry_time": current_data.name,
            "position": direction,
            "entry_price": entry_price,
            "exit_time": None,
            "exit_price": None,
            "pnl": None,
            "reason": None,
            "initial_capital": self._format_currency(self.initial_capital),
            "points": None,
            "profit": None,
            "win": None,
            "reward": None,
            "time_in_trade": None
        })

    def _close_position(self, current_data: pd.Series, reason: str, override_exit_price=None) -> float:
        if self.position.status == "flat":
            return 0.0

        direction = self.position.direction
        if override_exit_price is not None:
            exit_price = override_exit_price
        elif "SL" in reason:
            # trailing SL or normal SL
            if self.position.trailing_sl_price is not None:
                exit_price = self.position.trailing_sl_price
            else:
                if direction == 1:
                    exit_price = self.position.entry_price - self.position.sl_points
                else:
                    exit_price = self.position.entry_price + self.position.sl_points
        else:
            # Manual/Force/Reverse => exit on current open
            exit_price = current_data['open']

        realized_pnl = (exit_price - self.position.entry_price) * direction * self.lot_size

        # Return margin + realized PnL - transaction_cost
        margin_return = self.position.margin_used
        self.current_capital += (margin_return + realized_pnl - self.transaction_cost)

        trade_return = realized_pnl / self.initial_capital
        total_reward = trade_return

        points = (exit_price - self.position.entry_price) * direction
        formatted_profit = self._format_currency(realized_pnl)
        win = realized_pnl > 0

        for trade in reversed(self.trade_log):
            if trade["exit_time"] is None:
                trade.update({
                    "exit_time": current_data.name,
                    "exit_price": exit_price,
                    "pnl": realized_pnl,
                    "reason": reason,
                    "time_in_trade": self.position.time_in_trade,
                    "points": round(points, 2),
                    "profit": formatted_profit,
                    "win": win,
                    "reward": total_reward
                })
                break

        self.position.reset()
        return total_reward

    def _check_sl_hit(self, current_data: pd.Series):
        if self.position.status == "flat":
            return False, None

        direction = self.position.direction
        entry_price = self.position.entry_price
        target_points = self.position.target_points

        # Calculate target price
        if direction == 1:
            target_price = entry_price + target_points
        else:
            target_price = entry_price - target_points

        # If target is broken, update trailing SL up/down to at least the target
        if direction == 1:
            if current_data['high'] >= target_price:
                if self.position.trailing_sl_price is not None:
                    self.position.trailing_sl_price = max(self.position.trailing_sl_price, target_price)
        else:
            if current_data['low'] <= target_price:
                if self.position.trailing_sl_price is not None:
                    self.position.trailing_sl_price = min(self.position.trailing_sl_price, target_price)

        current_open = current_data['open']
        current_high = current_data['high']
        current_low = current_data['low']

        # Decide which SL to use if trailing is set
        if self.position.trailing_sl_price is not None:
            effective_sl = self.position.trailing_sl_price
        else:
            if direction == 1:
                effective_sl = entry_price - self.position.sl_points
            else:
                effective_sl = entry_price + self.position.sl_points

        # Check gap scenarios first
        if direction == 1:
            if current_open <= effective_sl:
                return True, current_open
            elif current_low <= effective_sl:
                return True, effective_sl
        else:
            if current_open >= effective_sl:
                return True, current_open
            elif current_high >= effective_sl:
                return True, effective_sl

        return False, None

    def _update_trailing_sl(self, current_data: pd.Series):
        if self.position.direction == 1:  # long
            if self.position.highest_price is None:
                self.position.highest_price = current_data['high']
            else:
                self.position.highest_price = max(self.position.highest_price, current_data['high'])

            new_sl_price = self._calculate_trailing_sl_price(
                reference_price=self.position.highest_price,
                side="long"
            )
            initial_sl_price = self.position.entry_price - self.position.sl_points
            self.position.trailing_sl_price = max(
                self.position.trailing_sl_price,
                new_sl_price,
                initial_sl_price
            )
        else:  # short
            if self.position.lowest_price is None:
                self.position.lowest_price = current_data['low']
            else:
                self.position.lowest_price = min(self.position.lowest_price, current_data['low'])

            new_sl_price = self._calculate_trailing_sl_price(
                reference_price=self.position.lowest_price,
                side="short"
            )
            initial_sl_price = self.position.entry_price + self.position.sl_points
            self.position.trailing_sl_price = min(
                self.position.trailing_sl_price,
                new_sl_price,
                initial_sl_price
            )

    def _calculate_trailing_sl_price(self, reference_price: float, side: str) -> float:
        if self.trailing_sl_mode == "fixed":
            if side == "long":
                return reference_price - self.trailing_sl_amount
            else:
                return reference_price + self.trailing_sl_amount

        elif self.trailing_sl_mode == "percentage":
            offset = reference_price * self.trailing_sl_percent
            if side == "long":
                return reference_price - offset
            else:
                return reference_price + offset

        elif self.trailing_sl_mode == "factor":
            if side == "long":
                gain = reference_price - self.position.entry_price
                return self.position.entry_price + (gain * self.trailing_factor)
            else:
                gain = self.position.entry_price - reference_price
                return self.position.entry_price - (gain * self.trailing_factor)

        else:
            return reference_price

    def _calculate_reward(self, unrealized_pnl: float, current_data: pd.Series) -> float:
        if self.position.sl_points == 0:
            risk_adj_pnl = 0.0
        else:
            risk = self.position.sl_points * self.lot_size
            risk_adj_pnl = (unrealized_pnl / risk) * self.unrealized_pnl_weight

        # Time penalty
        time_penalty = self.time_penalty_weight * (self.position.time_in_trade / 100.0)

        # Volatility penalty
        volatility = (current_data['high'] - current_data['low']) / max(current_data['open'], 1e-6)
        volatility_penalty = self.volatility_penalty_weight * volatility

        # SL proximity penalty
        if self.position.trailing_sl_price is not None:
            effective_sl = self.position.trailing_sl_price
        else:
            if self.position.direction == 1:
                effective_sl = self.position.entry_price - self.position.sl_points
            else:
                effective_sl = self.position.entry_price + self.position.sl_points

        if self.position.direction == 1:
            sl_range = abs(effective_sl - self.position.entry_price)
            sl_distance = max(0, current_data['close'] - effective_sl)
        else:
            sl_range = abs(effective_sl - self.position.entry_price)
            sl_distance = max(0, effective_sl - current_data['close'])

        if sl_range > 0:
            sl_proximity_ratio = min(sl_distance / sl_range, 1.0)
        else:
            sl_proximity_ratio = 1.0

        sl_proximity_penalty = 0.05 * (1.0 - sl_proximity_ratio)

        total_reward = risk_adj_pnl - time_penalty - volatility_penalty - sl_proximity_penalty
        return float(np.clip(total_reward, -3.0, 3.0))

    def _check_margin_call(self, current_price: float) -> bool:
        if self.position.status == "flat":
            return False

        maintenance_margin = self.position.margin_used * MARGIN_CALL_THRESHOLD
        unrealized_pnl = self.position.update_unrealized_pnl(current_price)
        equity = self.current_capital + unrealized_pnl

        if equity < maintenance_margin:
            return True
        return False

    def get_metrics(self) -> dict:
        if not self.trade_log:
            return {}

        closed_trades = [t for t in self.trade_log if t['exit_time'] is not None]
        total_trades = len(closed_trades)
        if total_trades == 0:
            return {"instrument": self.instrument_name, "message": "No trades taken"}

        winning_trades = [t for t in closed_trades if t['pnl'] is not None and t['pnl'] > 0]
        losing_trades = [t for t in closed_trades if t['pnl'] is not None and t['pnl'] <= 0]

        win_rate = (len(winning_trades) / total_trades) * 100.0
        sum_winning = sum(t['pnl'] for t in winning_trades)
        sum_losing = sum(t['pnl'] for t in losing_trades)
        abs_sum_losing = abs(sum_losing)

        if abs_sum_losing < 1e-10:
            profit_factor = float('inf')
        else:
            profit_factor = sum_winning / abs_sum_losing

        return {
            "instrument": self.instrument_name,
            "initial_capital": self._format_currency(self.initial_capital),
            "final_capital": self._format_currency(self.current_capital),
            "total_trades": total_trades,
            "win_rate": f"{round(win_rate, 2)}%",
            "total_reward": round(self.total_reward, 2),
            "max_drawdown": self._compute_max_drawdown(),
            "profit_factor": profit_factor
        }

    def _compute_max_drawdown(self) -> float:
        equity_curve = [self.initial_capital]
        for trade in self.trade_log:
            if trade['pnl'] is not None:
                equity_curve.append(equity_curve[-1] + trade['pnl'])
        equity_curve = np.array(equity_curve)
        if len(equity_curve) < 2:
            return 0.0
        peak = np.maximum.accumulate(equity_curve)
        dd = (peak - equity_curve) / peak
        return dd.max()

    def _format_currency(self, value: float) -> str:
        abs_value = abs(value)
        if abs_value >= 1e7:
            return f"₹{value / 1e7:.2f}Cr"
        elif abs_value >= 1e5:
            return f"₹{value / 1e5:.2f}L"
        elif abs_value >= 1e3:
            return f"₹{value / 1e3:.2f}K"
        else:
            return f"₹{value:.2f}"

In [22]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
import torch.multiprocessing as mp
import torch.utils.data as data
import torch.amp as amp
import torch.utils.checkpoint as cp

import numpy as np
import random
import time
import pickle
import os

# fix the checkpoint warning by explicitly specifying use_reentrant=False
def checkpoint_forward(module, x):
    return module(x)

model_base_config = {
    "d_model": 512, # 1024
    "nhead": 16, # 32
    "num_layers": 12, # 24
    "dim_feedforward": 2048, # 4096
    "dropout": 0.1,
    "instrument_embed_dim": 64, # 128

    "num_env_steps": 100000,
    "rollout_steps": 512,
    "grpo_epochs": 10,
    "mini_batch_size": 64,
    "clip_param": 0.2,
    "lr": 3e-4,
    "entropy_coef": 0.01,
    "max_grad_norm": 0.5,
    "weight_decay": 1e-5,

    "stochastic_depth_prob": 0.2,

    "use_gpu": True,
    "log_interval": 1,
    "eval_interval": 2500,

    "memory_file_path": "experience_memory.pkl",
    "max_memory_size": 50000,  # fixed memory limit
    "early_stopping_patience": 5,
}

device = torch.device("cuda" if (torch.cuda.is_available() and model_base_config["use_gpu"]) else "cpu")
print(device)

cuda


In [23]:
class StochasticDepth(nn.Module):
    def __init__(self, p: float = 0.2):
        super(StochasticDepth, self).__init__()
        self.p = p

    def forward(self, x, residual):
        if self.training and torch.rand(1).item() < self.p:
            return x
        else:
            return x + residual

In [24]:
class FFNFeedForward(nn.Module):
    def __init__(self, d_model, dim_feedforward, dropout):
        super(FFNFeedForward, self).__init__()
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

In [25]:
class MoEFeedForward(nn.Module):
    def __init__(self, d_model, dim_feedforward, dropout, num_experts=4):
        super(MoEFeedForward, self).__init__()
        self.num_experts = num_experts
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, dim_feedforward),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(dim_feedforward, d_model)
            ) for _ in range(num_experts)
        ])
        self.gate = nn.Linear(d_model, num_experts)
        self.softmax = nn.Softmax(dim=-1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, D = x.shape
        gate_scores = self.softmax(self.gate(x))
        out_experts = []
        for i in range(self.num_experts):
            out_i = self.experts[i](x)
            out_experts.append(out_i.unsqueeze(-1))
        out_experts = torch.cat(out_experts, dim=-1)
        gate_scores = gate_scores.unsqueeze(2)
        moe_output = (out_experts * gate_scores).sum(dim=-1)
        return moe_output

In [26]:
class MultiHeadLatentAttention(nn.Module):
    def __init__(self, d_model, nhead, dropout=0.1, num_latent_tokens=4):
        super(MultiHeadLatentAttention, self).__init__()
        self.mha = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.latent_tokens = nn.Parameter(torch.randn(num_latent_tokens, d_model))
        self.layernorm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, D = x.shape
        latent_tokens_expanded = self.latent_tokens.unsqueeze(0).expand(B, -1, -1)
        x_cat = torch.cat([latent_tokens_expanded, x], dim=1)
        attn_output, _ = self.mha(x_cat, x_cat, x_cat)
        x_out = self.layernorm(x_cat + self.dropout(attn_output))
        return x_out[:, self.latent_tokens.size(0):, :]

In [27]:
class DeepTransformerLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, dropout, stochastic_depth_prob, use_moe=False):
        super(DeepTransformerLayer, self).__init__()
        self.attn = MultiHeadLatentAttention(d_model, nhead, dropout)
        self.layernorm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)

        if use_moe:
            self.ffn = MoEFeedForward(d_model, dim_feedforward, dropout)
        else:
            self.ffn = FFNFeedForward(d_model, dim_feedforward, dropout)

        self.layernorm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)
        self.stochastic_depth = StochasticDepth(stochastic_depth_prob)

    def forward(self, x):
        attn_out = self.attn(x)
        x = self.stochastic_depth(
            x=self.layernorm1(x + self.dropout1(attn_out)),
            residual=x
        )
        ffn_out = self.ffn(x)
        x = self.stochastic_depth(
            x=self.layernorm2(x + self.dropout2(ffn_out)),
            residual=x
        )
        return x

In [28]:
def checkpoint_forward(module, x):
    return module(x)

class DeepTransformerCheckpointed(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, dropout,
                 total_layers=24, stochastic_depth_prob=0.2, use_checkpointing=True):
        super(DeepTransformerCheckpointed, self).__init__()
        self.layers = nn.ModuleList()
        self.use_checkpointing = use_checkpointing

        ffn_layers = 4
        moe_layers = total_layers - ffn_layers

        for _ in range(ffn_layers):
            self.layers.append(
                DeepTransformerLayer(
                    d_model, nhead, dim_feedforward, dropout,
                    stochastic_depth_prob=stochastic_depth_prob,
                    use_moe=False
                )
            )
        for _ in range(moe_layers):
            self.layers.append(
                DeepTransformerLayer(
                    d_model, nhead, dim_feedforward, dropout,
                    stochastic_depth_prob=stochastic_depth_prob,
                    use_moe=True
                )
            )

    def forward(self, x):
        for layer in self.layers:
            # Ensure x can compute gradients when checkpointing
            if self.use_checkpointing and self.training:
                if not x.requires_grad:
                    x = x.clone().detach().requires_grad_(True)
                x = cp.checkpoint(checkpoint_forward, layer, x, use_reentrant=False)
            else:
                x = layer(x)
        return x

In [29]:
class MultiEncoder(nn.Module):
    def __init__(self, obs_dim, model_config, num_instruments=1):
        super(MultiEncoder, self).__init__()
        self.instrument_embed_dim = model_config["instrument_embed_dim"]
        self.instrument_embedding = nn.Embedding(num_instruments, self.instrument_embed_dim)

        self.input_linear = nn.Linear(obs_dim + self.instrument_embed_dim, model_config["d_model"])

        self.transformer_block = DeepTransformerCheckpointed(
            d_model=model_config["d_model"],
            nhead=model_config["nhead"],
            dim_feedforward=model_config["dim_feedforward"],
            dropout=model_config["dropout"],
            total_layers=model_config["num_layers"],
            stochastic_depth_prob=model_config["stochastic_depth_prob"],
            use_checkpointing=True
        )

        self.policy_head = nn.Linear(model_config["d_model"], 4)
        self.obs_dim = obs_dim
        self.d_model = model_config["d_model"]

    def forward(self, obs_dict, instrument_id):
        history = obs_dict["history"]
        B, seq_len, feat_dim = history.shape
        inst_emb = self.instrument_embedding(instrument_id)
        inst_emb_expanded = inst_emb.unsqueeze(1).expand(B, seq_len, self.instrument_embed_dim)
        x = torch.cat([history, inst_emb_expanded], dim=-1)
        x = self.input_linear(x)
        x = self.transformer_block(x)
        final_repr = x[:, -1, :]
        policy_logits = self.policy_head(final_repr)
        return policy_logits

In [30]:
class ExternalMemory:
    def __init__(self, memory_file_path, max_memory_size=50000):
        self.memory_file_path = memory_file_path
        self.max_memory_size = max_memory_size
        self.experiences = []
        if os.path.exists(self.memory_file_path):
            with open(self.memory_file_path, 'rb') as f:
                self.experiences = pickle.load(f)

    def add_experience(self, obs, action, logprob, reward, done, instrument_id):
        self.experiences.append((obs, action, logprob, reward, done, instrument_id))
        # If we've exceeded the max limit, pop from the front
        if len(self.experiences) > self.max_memory_size:
            self.experiences.pop(0)

    def save_memory(self):
        with open(self.memory_file_path, 'wb') as f:
            pickle.dump(self.experiences, f)

    def clear_memory(self):
        self.experiences = []
        if os.path.exists(self.memory_file_path):
            os.remove(self.memory_file_path)

    def get_all_experiences(self):
        return self.experiences

In [31]:
class GRPORolloutBuffer:
    def __init__(self, rollout_size, device):
        self.rollout_size = rollout_size
        self.device = device
        self.reset()

    def reset(self):
        self.observations = []
        self.actions = []
        self.logprobs = []
        self.rewards = []
        self.dones = []
        self.instrument_ids = []
        self.advantages = []

    def add(self, obs, action, logprob, reward, done, instrument_id):
        self.observations.append(obs)
        self.actions.append(action)
        self.logprobs.append(logprob)
        self.rewards.append(reward)
        self.dones.append(done)
        self.instrument_ids.append(instrument_id)

    def compute_group_advantages(self):
        rewards_np = np.array(self.rewards, dtype=np.float32)
        mean_r = np.mean(rewards_np)
        adv = rewards_np - mean_r
        self.advantages = adv.tolist()

    def get_batches(self, mini_batch_size):
        indices = np.arange(len(self.observations))
        np.random.shuffle(indices)
        for start in range(0, len(self.observations), mini_batch_size):
            end = start + mini_batch_size
            batch_idx = indices[start:end]
            yield batch_idx

In [32]:
class GRPOTrainer:
    def __init__(self, model_config):
        self.model_config = model_config
        self.clip_param = model_config["clip_param"]
        self.grpo_epochs = model_config["grpo_epochs"]
        self.mini_batch_size = model_config["mini_batch_size"]
        self.entropy_coef = model_config["entropy_coef"]
        self.max_grad_norm = model_config["max_grad_norm"]
        self.rollout_steps = model_config["rollout_steps"]

        self.policy_model = None
        self.optimizer = None
        self.external_memory = ExternalMemory(
            model_config["memory_file_path"],
            max_memory_size=model_config["max_memory_size"]
        )
        self.early_stopping_patience = model_config["early_stopping_patience"]
        self.epochs_no_improve = 0

        self.train_logs = {
            "episode_rewards": [],
            "grpo_losses": [],
            "best_val_reward": -float("inf")
        }

        if device.type == "cuda":
            self.scaler = amp.GradScaler("cuda")
        else:
            self.scaler = amp.GradScaler(enabled=False)

    # Build the MultiEncoder once we know the obs_dim and #instruments
    def build_policy_model(self, obs_dim, num_instruments):
        self.policy_model = MultiEncoder(
            obs_dim=obs_dim,
            model_config=self.model_config,
            num_instruments=num_instruments
        ).to(device)
        self.optimizer = optim.Adam(
            self.policy_model.parameters(),
            lr=self.model_config["lr"],
            weight_decay=self.model_config["weight_decay"]
        )

    def _convert_obs(self, obs, instrument_id):
        obs_dict_t = {
            "history": torch.tensor(obs["history"], dtype=torch.float32, device=device).unsqueeze(0)
        }
        inst_id_t = torch.tensor([instrument_id], dtype=torch.long, device=device)
        return obs_dict_t, inst_id_t

    def _collect_rollout(self, env, instrument_id):
        buffer = GRPORolloutBuffer(self.rollout_steps, device)
        obs, _ = env.reset()
        done = False
        total_reward = 0.0

        for step in range(self.rollout_steps):
            obs_dict_t, inst_id_t = self._convert_obs(obs, instrument_id)
            with torch.no_grad():
                logits = self.policy_model(obs_dict_t, inst_id_t)
                dist = Categorical(logits=logits)
                action = dist.sample()
                logprob = dist.log_prob(action).item()

            action_int = action.item()
            next_obs, reward, done, _, _ = env.step(action_int)
            total_reward += reward

            buffer.add(obs, action_int, logprob, reward, done, instrument_id)
            self.external_memory.add_experience(obs, action_int, logprob, reward, done, instrument_id)

            obs = next_obs
            if done:
                obs, _ = env.reset()

        buffer.compute_group_advantages()
        return buffer, total_reward

    def _update_grpo(self, buffer):
        obs_arr = buffer.observations
        actions_arr = buffer.actions
        old_logprobs_arr = buffer.logprobs
        advantages_arr = buffer.advantages
        instrument_ids_arr = buffer.instrument_ids

        old_logprobs_tensor = torch.tensor(old_logprobs_arr, dtype=torch.float32, device=device)
        advantages_tensor = torch.tensor(advantages_arr, dtype=torch.float32, device=device)
        actions_tensor = torch.tensor(actions_arr, dtype=torch.long, device=device)
        advantages_tensor = (advantages_tensor - advantages_tensor.mean()) / (advantages_tensor.std() + 1e-8)

        total_policy_loss = 0.0
        total_entropy = 0.0

        for _ in range(self.grpo_epochs):
            for batch_idx in buffer.get_batches(self.mini_batch_size):
                obs_batch = [obs_arr[i] for i in batch_idx]
                actions_batch = actions_tensor[batch_idx]
                old_logprobs_batch = old_logprobs_tensor[batch_idx]
                advantages_batch = advantages_tensor[batch_idx]
                inst_id_batch_np = [instrument_ids_arr[i] for i in batch_idx]

                obs_dict_list = []
                inst_id_list = []
                for ob, inst_id in zip(obs_batch, inst_id_batch_np):
                    ob_dict_t, inst_id_t = self._convert_obs(ob, inst_id)
                    obs_dict_list.append(ob_dict_t)
                    inst_id_list.append(inst_id_t)

                history_batch = torch.cat([o["history"] for o in obs_dict_list], dim=0)
                inst_id_batch_tensor = torch.cat(inst_id_list, dim=0)
                batch_obs_dict = {"history": history_batch}

                with amp.autocast(device_type=device.type, enabled=(device.type=="cuda")):
                    logits = self.policy_model(batch_obs_dict, inst_id_batch_tensor)
                    dist = Categorical(logits=logits)
                    logprobs = dist.log_prob(actions_batch)
                    entropy = dist.entropy().mean()

                    ratio = torch.exp(logprobs - old_logprobs_batch)
                    surr1 = ratio * advantages_batch
                    surr2 = torch.clamp(ratio, 1.0 - self.clip_param, 1.0 + self.clip_param) * advantages_batch
                    policy_loss = -torch.min(surr1, surr2).mean()
                    loss = policy_loss - self.entropy_coef * entropy

                self.optimizer.zero_grad()
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                nn.utils.clip_grad_norm_(self.policy_model.parameters(), self.max_grad_norm)
                self.scaler.step(self.optimizer)
                self.scaler.update()

                total_policy_loss += policy_loss.item()
                total_entropy += entropy.item()

        n_updates = self.grpo_epochs * max(1, (len(obs_arr) // self.mini_batch_size))
        avg_policy_loss = total_policy_loss / max(1, n_updates)
        avg_entropy = total_entropy / max(1, n_updates)
        return avg_policy_loss, avg_entropy

    def evaluate(self, env, instrument_id, n_eval_episodes=3):
        if os.path.exists("best_grpo_model.pt"):
            self.load_checkpoint("best_grpo_model.pt")

        total_eval_reward = 0.0
        for _ in range(n_eval_episodes):
            obs, _ = env.reset()
            done = False
            episode_reward = 0.0
            while not done:
                obs_dict_t, inst_id_t = self._convert_obs(obs, instrument_id)
                with torch.no_grad():
                    logits = self.policy_model(obs_dict_t, inst_id_t)
                    dist = Categorical(logits=logits)
                    action = dist.sample().item()
                obs, reward, done, _, _ = env.step(action)
                episode_reward += reward
            total_eval_reward += episode_reward
        return total_eval_reward / n_eval_episodes

    def save_checkpoint(self, path):
        checkpoint = {
            "model_state_dict": self.policy_model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "train_logs": self.train_logs
        }
        torch.save(checkpoint, path)

    def load_checkpoint(self, path):
        checkpoint = torch.load(path, map_location=device)
        if self.policy_model is not None:
            self.policy_model.load_state_dict(checkpoint["model_state_dict"])
        if self.optimizer is not None:
            self.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        self.train_logs = checkpoint["train_logs"]

    def train_one_chunk(self, envs, steps_for_this_chunk):
        steps_collected = 0
        rollout_count = 0

        while steps_collected < steps_for_this_chunk:
            rollout_count += 1
            env_index = np.random.randint(len(envs))
            env = envs[env_index]

            buffer, rollout_reward = self._collect_rollout(env, instrument_id=env_index)
            steps_collected += self.rollout_steps

            policy_loss, entropy = self._update_grpo(buffer)
            self.train_logs["grpo_losses"].append((policy_loss, entropy))
            self.train_logs["episode_rewards"].append(rollout_reward)

            metrics = env.get_metrics() if hasattr(env, "get_metrics") else {}
            if rollout_count % self.model_config["log_interval"] == 0:
                print(f"Rollout: {rollout_count}, Steps in chunk: {steps_collected}, "
                      f"Instrument: {metrics.get('instrument','Unknown')}, Reward: {rollout_reward:.2f}, "
                      f"InitCap: {metrics.get('initial_capital','N/A')}, "
                      f"FinalCap: {metrics.get('final_capital','N/A')}, "
                      f"Trades: {metrics.get('total_trades',0)}, WinRate: {metrics.get('win_rate','N/A')}, "
                      f"PolicyLoss: {policy_loss:.4f}, Entropy: {entropy:.4f}")

            if (self.model_config["eval_interval"] > 0) and (rollout_count % self.model_config["eval_interval"] == 0):
                val_index = np.random.randint(len(envs))
                val_env = envs[val_index]
                val_reward = self.evaluate(val_env, val_index, n_eval_episodes=3)
                val_metrics = val_env.get_metrics() if hasattr(val_env, "get_metrics") else {}
                print(f"[Validation] Instrument: {val_metrics.get('instrument','Unknown')}, "
                      f"AvgReward: {val_reward:.2f}, InitCap: {val_metrics.get('initial_capital','N/A')}, "
                      f"FinalCap: {val_metrics.get('final_capital','N/A')}, "
                      f"Trades: {val_metrics.get('total_trades',0)}, WinRate: {val_metrics.get('win_rate','N/A')}")

                if val_reward > self.train_logs["best_val_reward"]:
                    self.train_logs["best_val_reward"] = val_reward
                    self.epochs_no_improve = 0
                    self.save_checkpoint("best_grpo_model.pt")
                    print(f"New best reward: {val_reward:.2f}. Checkpoint saved.")
                else:
                    self.epochs_no_improve += 1
                    if self.epochs_no_improve >= self.early_stopping_patience:
                        print(f"Early stopping triggered! No improvement for {self.early_stopping_patience} evaluations.")
                        return steps_collected
        return steps_collected

    def train_in_chunks(self, global_config, chunk_size=10, total_env_steps=None, create_env_fn=None):
        if total_env_steps is None:
            total_env_steps = self.model_config["num_env_steps"]
        full_instrument_list = global_config["instruments"]

        def chunkify(lst, chunk_sz):
            for i in range(0, len(lst), chunk_sz):
                yield lst[i:i + chunk_sz]

        instrument_chunks = list(chunkify(full_instrument_list, chunk_size))

        if create_env_fn is None:
            raise ValueError("Please provide 'create_env_fn' to build your environment.")
        if len(instrument_chunks) == 0 or len(instrument_chunks[0]) == 0:
            raise ValueError("No instrument configs found in global_config['instruments']")

        # Build a sample env to determine obs_dim
        first_inst_conf = instrument_chunks[0][0]
        merged_conf_sample = global_config.copy()
        merged_conf_sample.update(first_inst_conf)
        sample_env = create_env_fn(merged_conf_sample)
        sample_obs, _ = sample_env.reset()
        obs_dim = sample_obs["history"].shape[-1]

        # Build the policy model
        self.build_policy_model(obs_dim, num_instruments=len(full_instrument_list))

        # If checkpoint exists, load it and resume
        if os.path.exists("best_grpo_model.pt"):
            self.load_checkpoint("best_grpo_model.pt")
            print("Checkpoint loaded. Resuming training from best_grpo_model.pt")

        steps_done = 0
        chunk_index = 0

        while steps_done < total_env_steps:
            for chunk_instruments in instrument_chunks:
                envs_for_chunk = []
                for inst_conf in chunk_instruments:
                    env_config = global_config.copy()
                    env_config.update(inst_conf)
                    envs_for_chunk.append(create_env_fn(env_config))

                steps_for_this_chunk = min(
                    self.rollout_steps * len(chunk_instruments) * 3,
                    total_env_steps - steps_done
                )

                chunk_steps_collected = self.train_one_chunk(envs_for_chunk, steps_for_this_chunk)
                steps_done += chunk_steps_collected
                print(f"Chunk {chunk_index} done: collected {chunk_steps_collected} steps, total so far {steps_done}")

                del envs_for_chunk
                if steps_done >= total_env_steps:
                    break
                chunk_index += 1

            if steps_done < total_env_steps:
                print(f"Completed full pass of chunks. Steps done: {steps_done}/{total_env_steps}.")
                chunk_index = 0

        self.external_memory.save_memory()
        print(f"Training finished. Total steps done: {steps_done}")

In [33]:
def create_env_fn(env_config):
  return TradingEnv(env_config)

trainer = GRPOTrainer(model_base_config)

trainer.train_in_chunks(
  global_config=config,
  chunk_size=10,
  total_env_steps=model_base_config["num_env_steps"],
  create_env_fn=create_env_fn
)

Rollout: 1, Steps in chunk: 512, Instrument: ASIAN_PAINTS_LTD_240M, Reward: -214.24, InitCap: ₹1.79K, FinalCap: ₹151.03, Trades: 47, WinRate: 57.45%, PolicyLoss: 0.4445, Entropy: 0.0588
Rollout: 2, Steps in chunk: 1024, Instrument: ADANI_PORTS_&_SEZ_LTD_240M, Reward: -238.06, InitCap: ₹810.70, FinalCap: ₹28.58, Trades: 19, WinRate: 52.63%, PolicyLoss: 0.1004, Entropy: 0.0105
Rollout: 3, Steps in chunk: 1536, Instrument: ADANI_ENTERPRISES_LTD_240M, Reward: -233.98, InitCap: ₹2.10K, FinalCap: ₹25.62, Trades: 48, WinRate: 39.58%, PolicyLoss: 0.0052, Entropy: 0.0135
Rollout: 4, Steps in chunk: 2048, Instrument: ADANI_ENTERPRISES_LTD_240M, Reward: -231.31, InitCap: ₹2.10K, FinalCap: ₹25.62, Trades: 48, WinRate: 39.58%, PolicyLoss: 0.0350, Entropy: 0.0004
Rollout: 5, Steps in chunk: 2560, Instrument: ADANI_ENTERPRISES_LTD_240M, Reward: -233.98, InitCap: ₹2.10K, FinalCap: ₹25.62, Trades: 48, WinRate: 39.58%, PolicyLoss: 0.0000, Entropy: 0.0002
Rollout: 6, Steps in chunk: 3072, Instrument: BAJ

RuntimeError: mat1 and mat2 shapes cannot be multiplied (30x77 and 78x512)